In [5]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

load_dotenv()

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why parrots talk")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots can talk. Let me start by recalling what I know about parrots. They\'re birds known for mimicking sounds, including human speech. But how exactly do they do that? I think it has to do with their vocal anatomy. Maybe their syrinx is different from other birds? Wait, what\'s a syrinx? Oh right, it\'s the vocal organ in birds, similar to the human larynx. Parrots might have a more complex syrinx that allows them to produce a wider range of sounds.\n\nAlso, I remember reading that some parrots can learn to associate words with meanings, which is more than just mimicry. But how common is that? Maybe not all parrots can talk, just certain species. The user mentioned "parrots" in general, so I should clarify if all species can talk or only some. For example, African Grey Parrots are famous for their talking abilities, like the one named Alex who could use words to identify objects. But other parrots might not be as articula

In [6]:
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """
    Get the weather of a location
    """
    return f"It's Rainy at {location} and it's not preffered to go out."

model_with_tools = model.bind_tools([get_weather])
model_with_tools

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000028D97963250>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000028D97963C50>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather of a location', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [7]:
# Invoking using tool
response = model_with_tools.invoke("What is the weather in boston")
print(response)
for tool_call in response.tool_calls:
    print(f"""Name of the tool: {tool_call['name']}
Location: {tool_call['args']['location']}""")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools provided. There\'s a function called get_weather that takes a location parameter. The required parameter is location, which should be a string. Since the user specified "boston" as the location, I need to call get_weather with location set to "Boston". I should make sure the JSON is correctly formatted with the name of the function and the arguments. Let me structure the tool_call accordingly.\n', 'tool_calls': [{'id': 'tw7nyybsc', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 153, 'total_tokens': 273, 'completion_time': 0.181917871, 'completion_tokens_details': {'reasoning_tokens': 96}, 'prompt_time': 0.007132492, 'prompt_tokens_details': None, 'queue_time': 0.047040818, 'total_time': 0.189050363}, 'model_name': 'qwen/qwen3-32b', 'sy

In [10]:
# Tool execution
# Step 1: Model generates tool calls
messages = [{"role":"user", "content": "What is the weather in boston"}]
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)

# Step 2: Execute tools and collect results
for tool_call in ai_message.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is currently rainy, and it's not recommended to go out. You might want to stay indoors or carry an umbrella if you need to be outside.
